# A Verified 22.68 Portfolio for Verifiable Quantum Circuit Compilation

**Public score: 22.68** (geometric mean across the three tasks), assembled from
component circuits that each pass the competition's official verifier.

This notebook does three things:

1. Lays out the scoring model the verifiers actually implement. Most of the
   leverage in this competition comes from understanding where cost hides.
2. Hash-verifies and assembles the best verified public circuit for each task
   into a valid `submission.zip`. The output can be submitted directly.
3. Documents provenance. Every component is Apache-2.0 public work from the
   Kaggle community; the contribution here is verification discipline,
   integration, and a map of where the remaining headroom sits.

---

## 1. The scoring model

The score is a **geometric mean of three improvement ratios** (baseline cost
divided by submission cost). Because the score multiplies the three ratios,
a 10% cost cut on the weakest task is worth exactly as much as 10% on the
strongest. The weakest task is where the effort pays.

| Task | Cost model | What dominates | Public best |
|---|---|---|---|
| **invsqrt** | `Q x (Toffoli + rotation cost)` | lookup size k and Horner depth | ~20.2x |
| **dqi** | `repetition x Q^2 x T` | qubit count enters squared | ~58.9x |
| **dfthcsos** | `lambda_eff x T_total x Q_total` | lambda ~47-50, T ~17k, Q ~430 | ~9.65x |

Three consequences that drove every decision here:

* **Qubits are quadratic on dqi.** At the margin, removing a qubit beats
  removing a Toffoli roughly 2:1. The 889-qubit Schur-family decoder with
  ~434k executed Toffolis outscores narrower decoders with fewer Toffolis
  but more qubits.
* **The repetition factor punishes observed failures.** When verification
  shots fail, the cost inflates by `repetition > 1`. We measured a candidate
  dqi circuit with fewer Toffolis but 1 value error in 102,400 shots, and it
  scored worse than a zero-error circuit with more Toffolis. A slightly
  cheaper errorful circuit is a worse submission.
* **Scratch qubits must return to |0> and phases are checked.** The verifier
  tracks measurement-induced phase flips (`HMR`/`R` applied to a non-zero
  qubit flips the output sign with probability 1/2). Cleanup is part of the
  score, not an afterthought.

## 2. Component portfolio (all figures are official-verifier outputs)

| Task | Source | Qubits | Verified cost | Ratio |
|---|---|---|---|---|
| invsqrt | busyaprime/mrmeows merge-SAT line (63q, C=1334.5) | 63 | 102,664 | **20.2x** |
| dqi | heliosli/SomeJoint confluent Schur-family decoder | 889 | 4.148e11 | **58.9x** |
| dfthcsos | somejoint R8/B143/C62 fit + Nilanjan column gauge, J=5 loaders | 430 | 3.477e8 | **9.65x** |

Geometric mean = (20.2 x 58.9 x 9.65)^(1/3) = **22.68**, matching the
leaderboard score returned for this exact file set.

Each `*.verify.json` shipped in the attached dataset is the unmodified JSON
written by the competition's own verifier: shots run, error bounds, Toffoli
counts, qubit counts, and the circuit SHA-256. The assembly cell below
re-checks every file against those digests before zipping, so what gets
submitted is byte-identical to what was verified.

## 3. Assemble `submission.zip` (hash-verified)

The cell locates the three components anywhere under `/kaggle/input`,
verifies each file's SHA-256 against the digests recorded at verification
time, and writes a competition-valid `submission.zip` to `/kaggle/working`.
Any mismatch aborts the build instead of shipping a bad archive.

In [ ]:
import hashlib, json, shutil, zipfile
from pathlib import Path

# Digests recorded from the official verifier runs on this exact file set.
EXPECTED = {
    "dqi.kmb": "4ee7a17b4cbca4053a19bc951e7f2ef91fabdbb036006c9d658adcff4fe76da4",
    "invsqrt.kmb": "07314803a0925970cf45007ebbdb8a146378c242e3991c70d6f7509af3d583da",
    "dfthcsos/angle_k_0.kmb": "5255d0d66795e2fa1e1e1b36416237e44cf21aad2e5479c40f87e1969403eca2",
    "dfthcsos/angle_k_1.kmb": "d262661e351a90a223d626a28aaa102d8431dad11bdb383dba356e4185aabfcd",
    "dfthcsos/angle_k_10.kmb": "bd98550e656553014cb1a13cae825666c5e68a0e7a384d5965107599f128e0f0",
    "dfthcsos/angle_k_11.kmb": "dd0551f1da64eec6dc73ec7688ea7f8c6418a216b8a9125eea486a2cf939ffd2",
    "dfthcsos/angle_k_12.kmb": "c34e9459efe42c4b3679d4e2a66ffd3b6556965fd462f5f2675772cd529a26f3",
    "dfthcsos/angle_k_13.kmb": "3e9a4e1b764043e22f3587675d2b578aca04df75425b3a66615e1a1088be600f",
    "dfthcsos/angle_k_14.kmb": "8bf80f72edf85122666c68a6e8af3d0a882003836478bc66418cbc87bac1a135",
    "dfthcsos/angle_k_2.kmb": "37d30f21a92aec50287ae7085b84d2cd7a490a1c961decfafe39b9b5c6776f2e",
    "dfthcsos/angle_k_3.kmb": "f3fce956a2a213bd650687cf8d329fceab2327c19cf43bacd05dddccf73882a0",
    "dfthcsos/angle_k_4.kmb": "1f949ad54757dfaafb6bfdac99894bffd0410d924b0e454d1aec3d894dd5afbd",
    "dfthcsos/angle_k_5.kmb": "84655d0b97912178da383df083a3424c3822f23e590abc95dcaaea0e998c2e79",
    "dfthcsos/angle_k_6.kmb": "27875555d228d50e3681676b63d84d638750afb4245f23a71b67d2f6a2cff1c5",
    "dfthcsos/angle_k_7.kmb": "49e43cd23195ec70278ca2d0e618c585b8a49189d4584ee59bd0a793a720f4fb",
    "dfthcsos/angle_k_8.kmb": "0e49460864fbb5270b8755d98e3d5af14b1b94d35d183dc842d1953253cfe845",
    "dfthcsos/angle_k_9.kmb": "20b1d32646f1965669ddd5c310779d7a0b23cfdf6276fdb5a415a75359c7fc09",
    "dfthcsos/submission.npz": "7213867fe0b4bb69dd3a5c9e7961ec442bbcce2f420b12c51f8bac7f6db1ee75"
}

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

WORK = Path("/kaggle/working")
SUB  = WORK / "submission"
if SUB.exists():
    shutil.rmtree(SUB)

# --- locate inputs anywhere under /kaggle/input ---
by_name = {}
for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        by_name.setdefault(p.name, p)

missing, bad = [], []
print("verifying component digests:")
for rel, want in EXPECTED.items():
    name = rel.split("/")[-1]
    src = by_name.get(name)
    if src is None:
        missing.append(rel); continue
    got = sha256(src)
    status = "ok  " if got == want else "FAIL"
    print(f"  {status} {rel:28s} {got[:16]}...")
    if got != want:
        bad.append((rel, want, got))
        continue
    dest = SUB / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dest)

if missing:
    print("\nMISSING (attach the portfolio dataset):", missing)
if bad:
    print("\nHASH MISMATCHES - refusing to build:", [b[0] for b in bad])
if missing or bad:
    raise SystemExit("assembly aborted: inputs did not match verified digests")

zpath = WORK / "submission.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(SUB.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(SUB))

names = zipfile.ZipFile(zpath).namelist()
print(f"\nwrote {zpath} ({zpath.stat().st_size/1e6:.1f} MB)")
assert "invsqrt.kmb" in names and "dqi.kmb" in names and "dfthcsos/submission.npz" in names
print("layout OK - ready to submit")

## 4. Verification records

The unmodified verifier outputs bundled with the dataset. Note the dqi
entry: `value_errors: 0`, `phase_errors: 0` across 102,400 shots. That
zero-error property is what the repetition-factor math rewards.

In [ ]:
import json
from pathlib import Path

for jf in Path("/kaggle/input").rglob("*.verify.json"):
    rec = json.loads(jf.read_text())
    keep = {k: rec.get(k) for k in (
        "status","score","num_qubits","toffoli_count","verified_shots",
        "value_errors","phase_errors","repetition_factor",
        "lambda_eff","error_1","error_2","circuit_sha256") if k in rec}
    print(jf.name); print(json.dumps(keep, indent=2)[:1400]); print()

# implied geometric mean from the verified costs
BASE = {"invsqrt": 2_076_774.0, "dqi": 2.4425e13, "dfthcsos": 3.355599588e9}
try:
    scores = {}
    for jf in Path("/kaggle/input").rglob("*.verify.json"):
        rec = json.loads(jf.read_text())
        for task in BASE:
            if jf.name.startswith(task) and rec.get("status") == "PASS":
                scores[task] = BASE[task] / rec["score"]
    if len(scores) == 3:
        gm = (scores["invsqrt"] * scores["dqi"] * scores["dfthcsos"]) ** (1/3)
        print(f"implied GM from verifier records: {gm:.2f}x "
              f"(invsqrt {scores['invsqrt']:.2f} x dqi {scores['dqi']:.2f} x chem {scores['dfthcsos']:.2f})")
except Exception as e:
    print("gm calc skipped:", e)

## 5. Notes from the verification work

A few things that only show up once you run the verifiers at scale:

* **Measurement-based uncomputation is not free.** `HMR` on a qubit holding
  `|1>` injects a coin-flip sign flip, and the simulator tracks it. Erasure
  on live data needs reversible uncompute or a matched compute/delete pair.
* **Re-encoded angle words are not idempotent.** Decoding and re-quantizing
  `angle_words` flips most words and can destroy a tuned fit. Carry the
  quantized state forward; never round-trip through decoded vectors.
* **Cost-model cliffs are real.** On dfthcsos, `B >= 129` drops inner-prep
  temporary qubits from 575 to 304, while `brot >= 9` doubles the angle
  register cost. Shape `(R, B)` fixes word count and therefore T; C is
  approximation slack.
* **`Circuit.to_file` writes text `.kmx` by default.** Binary submissions
  need `format='kmb'`; text files named `.kmb` are rejected.

## 6. Provenance and license

Components are assembled from Apache-2.0 public releases by the community.
Credit goes to the invsqrt merge/SAT-compiled line (busyaprime/mrmeows), the
SomeJoint/heliosli confluent DQI decoder family, and the somejoint
R8/B143/C62 FeMoCo factorization with Nilanjan's column gauge. Per-file
digests in the verifier records identify exactly which artifacts are used.
The value added here is verification discipline, integration, and the
scoring-model map in section 1.